# Concordance 03 — Dynamo

Standalone Dynamo run on the bone marrow CD34+ dataset.

**Environment.** `scvelo + dynamo-release`. Run in a fresh env
separate from scjdo and splicejac:

    conda create -n dynamo python=3.10 -y && conda activate dynamo
    pip install scvelo dynamo-release


In [1]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd
SEED   = 0
N_PCS  = 30
TOP_K  = 30
RESULTS_DIR = Path('concordance_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)


## 1. Load + shared preprocessing

In [2]:
import scvelo as scv
import scanpy as sc
import numpy as np

adata = scv.datasets.bonemarrow()
scv.pp.filter_and_normalize(adata, min_shared_counts=20)
scv.pp.moments(adata, n_pcs=N_PCS, n_neighbors=30)
sc.tl.diffmap(adata, n_comps=15)

# Deterministic iroot — highest-CD34 cell in HSC_1
_label_col = next((c for c in ['clusters','celltype','cell_type','leiden']
                   if c in adata.obs), None)
_hsc_mask  = adata.obs[_label_col].astype(str).str.startswith('HSC').to_numpy() \
             if _label_col else np.ones(adata.n_obs, bool)
if 'CD34' in adata.var_names:
    _cd34 = adata[:, 'CD34'].X
    _cd34 = np.asarray(_cd34.todense()).ravel() if hasattr(_cd34, 'todense') \
            else np.asarray(_cd34).ravel()
else:
    _cd34 = np.zeros(adata.n_obs)
if _hsc_mask.any() and _cd34.max() > 0:
    _idx = np.flatnonzero(_hsc_mask)
    adata.uns['iroot'] = int(_idx[np.argmax(_cd34[_idx])])
else:
    adata.uns['iroot'] = int(np.argmax(_cd34)) if _cd34.max() > 0 else 0
sc.tl.dpt(adata)
adata.obs['pseudotime'] = adata.obs['dpt_pseudotime'].astype(np.float32)
pt = adata.obs['pseudotime'].to_numpy()
adata.obs['pseudotime'] = ((pt - np.nanmin(pt)) /
                            (np.nanmax(pt) - np.nanmin(pt) + 1e-9)).astype(np.float32)
adata.obs['cluster'] = adata.obs[_label_col].astype('category')
CLUSTERS   = adata.obs['cluster'].cat.categories.tolist()
GENE_NAMES = list(adata.var_names)
print(f'{adata.n_obs} cells × {adata.n_vars} genes  | '
      f'clusters: {CLUSTERS}  | iroot={adata.uns["iroot"]}')


Filtered out 7837 genes that are detected 20 counts (shared).
Normalized count data: X, spliced, unspliced.
Logarithmized X.


/Users/terooatt/miniconda3/lib/python3.13/site-packages/scvelo/preprocessing/utils.py:705: DeprecationWarning: `log1p` is deprecated since scVelo v0.3.0 and will be removed in a future version. Please use `log1p` from `scanpy.pp` instead.
  log1p(adata)
/var/folders/8q/0m_1_8yj0r1_hxyz4br3vg4r0000gp/T/ipykernel_45018/1859992275.py:7: DeprecationWarning: Automatic neighbor calculation is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors first with Scanpy.
  scv.pp.moments(adata, n_pcs=N_PCS, n_neighbors=30)
/Users/terooatt/miniconda3/lib/python3.13/site-packages/scvelo/preprocessing/moments.py:71: DeprecationWarning: `neighbors` is deprecated since scvelo==0.4.0 and will be removed in a future version of scVelo. Please compute neighbors with Scanpy.
  neighbors(
/Users/terooatt/miniconda3/lib/python3.13/site-packages/scvelo/preprocessing/neighbors.py:233: DeprecationWarning: Automatic computation of PCA is deprecated since scvelo=

computing neighbors


    finished (0:00:03) --> added 
    'distances' and 'connectivities', weighted adjacency matrices (adata.obsp)
computing moments based on connectivities


    finished (0:00:01) --> added 
    'Ms' and 'Mu', moments of un/spliced abundances (adata.layers)
5780 cells × 6482 genes  | clusters: ['HSC_1', 'HSC_2', 'Ery_1', 'Mono_1', 'Precursors', 'CLP', 'Mono_2', 'DCs', 'Ery_2', 'Mega']  | iroot=1722


In [3]:
import json
metadata = {
    'dataset':         'scvelo.datasets.bonemarrow (Setty 2019 CD34+)',
    'n_cells':         int(adata.n_obs),
    'n_genes':         int(adata.n_vars),
    'seed':            SEED,
    'top_k':           TOP_K,
    'n_pcs':           N_PCS,
    'cluster_order':   CLUSTERS,
    'iroot_cell_idx':  int(adata.uns['iroot']),
}
(RESULTS_DIR / 'shared_metadata.json').write_text(json.dumps(metadata, indent=2))
print('shared_metadata.json saved')


shared_metadata.json saved


## 2. Dynamo — per-cell Jacobian aggregated to per-cluster centroid

In [4]:
import dynamo as dyn
adata_dyn = adata.copy()
dyn.pp.recipe_monocle(adata_dyn)
dyn.tl.dynamics(adata_dyn, model='stochastic', cores=1)
dyn.tl.reduceDimension(adata_dyn)
dyn.tl.cell_velocities(adata_dyn, basis='pca')
dyn.vf.VectorField(adata_dyn, basis='pca', M=100, restart_num=1)
dyn.vf.jacobian(adata_dyn, basis='pca')

Jcell  = adata_dyn.uns['jacobian_pca']['jacobian']     # (d, d, n_cells)
PCs    = adata_dyn.varm['PCs']                          # (n_var, n_pcs)

inst_arr       = np.full(len(CLUSTERS), np.nan, dtype=np.float32)
gene_vec       = np.zeros((len(CLUSTERS), len(GENE_NAMES)), dtype=np.float32)
gene_inst_rank = np.zeros_like(gene_vec)
top_genes      = {}
for i, c in enumerate(CLUSTERS):
    sel = (adata_dyn.obs['cluster'] == c).to_numpy()
    if sel.sum() < 5: continue
    Jc = Jcell[..., sel].mean(-1)
    w, V = np.linalg.eig(Jc); k = int(np.argmax(w.real))
    inst_arr[i] = float(w[k].real)
    v_lat = V[:, k].real
    v_lat = v_lat / (np.linalg.norm(v_lat) + 1e-12)
    g = PCs[:, :len(v_lat)] @ v_lat
    g = g / (np.linalg.norm(g) + 1e-12)
    # Re-index to the SHARED GENE_NAMES order (dynamo may have
    # filtered some genes during recipe_monocle).
    g_full = np.zeros(len(GENE_NAMES), dtype=np.float32)
    name_to_g = {n: gv for n, gv in zip(adata_dyn.var_names, g)}
    for j, gn in enumerate(GENE_NAMES):
        if gn in name_to_g: g_full[j] = float(name_to_g[gn])
    gene_vec[i]       = g_full / (np.linalg.norm(g_full) + 1e-12)
    gene_inst_rank[i] = np.abs(g_full)
    top_idx = np.argsort(gene_inst_rank[i])[::-1][:TOP_K]
    top_genes[c] = np.array([GENE_NAMES[j] for j in top_idx])
print(pd.Series({c: inst_arr[i] for i, c in enumerate(CLUSTERS)
                if not np.isnan(inst_arr[i])})
      .sort_values(ascending=False).round(3))

|-----? dynamo.preprocessing.deprecated is deprecated.


|-----> recipe_monocle_keep_filtered_cells_key is None. Using default value from DynamoAdataConfig: recipe_monocle_keep_filtered_cells_key=True


|-----> recipe_monocle_keep_filtered_genes_key is None. Using default value from DynamoAdataConfig: recipe_monocle_keep_filtered_genes_key=True


|-----> recipe_monocle_keep_raw_layers_key is None. Using default value from DynamoAdataConfig: recipe_monocle_keep_raw_layers_key=True


|-----> apply Monocole recipe to adata...


...

|-----------> method ball_tree selected


|-----> [SparseVFC] completed [0.0954s]


|-----> [VectorField] completed [0.1659s]



╭─ SUMMARY: VectorField ─────────────────────────────────────────────╮
│  Duration: 0.1672s                                                 │
│  Shape:    5,780 x 6,482 (Unchanged)                               │
│                                                                    │
│  CHANGES DETECTED                                                  │
│  ────────────────                                                  │
│  ● OBS    │ ✚ control_point_pca (bool)                             │
│           │ ✚ inlier_prob_pca (float)                              │
│           │ ✚ obs_vf_angle_pca (float)                             │
│                                                                    │
│  ● UNS    │ ✚ VecFld_pca                                           │
│                                                                    │
│  ● OBSM   │ ✚ X_pca_SparseVFC (array, 5780x30)                     │
│           │ ✚ velocity_pca_SparseVFC (array, 5780x30)              │
│    


╭─ SUMMARY: jacobian ────────────────────────────────────────────────╮
│  Duration: 0.1584s                                                 │
│  Shape:    5,780 x 6,482 (Unchanged)                               │
│                                                                    │
│  CHANGES DETECTED                                                  │
│  ────────────────                                                  │
│  ● OBS    │ ✚ jacobian_det_pca (float)                             │
│                                                                    │
│  ● UNS    │ ✚ jacobian_pca                                         │
│                                                                    │
╰────────────────────────────────────────────────────────────────────╯
Precursors    0.048
Mono_2        0.040
HSC_2         0.036
Ery_2         0.030
Ery_1         0.030
Mono_1        0.015
DCs           0.015
Mega          0.012
HSC_1         0.009
CLP           0.004
dtype: float32


## 3. Save standardized .npz

In [5]:
save_path = RESULTS_DIR / 'dynamo_per_cluster.npz'
np.savez_compressed(save_path,
    method='dynamo',
    clusters=np.array(CLUSTERS),
    inst_per_cluster=inst_arr,
    gene_names=np.array(GENE_NAMES),
    gene_vec=gene_vec,
    gene_inst_rank=gene_inst_rank,
    **{f'top_genes/{c}': top_genes[c] for c in top_genes})
print(f'Saved → {save_path}  ({save_path.stat().st_size:,} bytes)')

Saved → concordance_results/dynamo_per_cluster.npz  (527,347 bytes)
